In [6]:
"""
March Mania 2026 - XGBoost Model with Seeds
=============================================
Features:
  - All box score stats (Off + Def averages) from regular season
  - WinPct
  - Seed Number          ← explicitly included
  - Seed Difference      ← T1 seed minus T2 seed
  - Upset Flag           ← 1 if lower seed (worse team) is favoured by seed

Train : 2013-2022 (regular season + tournament games)
Test  : 2023 NCAA Tournament
No data leakage — all features built from regular season only, season-scoped.
"""

import os, re, warnings
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import log_loss, accuracy_score, brier_score_loss

warnings.filterwarnings("ignore")

# ─────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print(f"Regular Season rows : {len(m_regular_season_detailed):,}")
print(f"Seeds rows          : {len(m_seeds):,}")
print(f"Tournament rows     : {len(m_tourney_detailed_result):,}")

# ─────────────────────────────────────
# 2. PARSE SEEDS
#    e.g. "W01" → 1,  "Y16a" → 16
# ─────────────────────────────────────
def parse_seed(s):
    digits = re.search(r'\d+', str(s))
    return int(digits.group()) if digits else 16

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)
seed_lookup = m_seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()

# ─────────────────────────────────────
# 3. BUILD TEAM SEASON STATS
#    Source: regular season only → no leakage
#    Off_ = team's own stats per game
#    Def_ = opponent's stats per game (what we allowed)
# ─────────────────────────────────────
STAT_COLS = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3',
             'FTM', 'FTA', 'OR', 'DR',
             'Ast', 'TO', 'Stl', 'Blk', 'PF']

def build_team_stats(reg):
    # Winner rows
    w            = reg[['Season','WTeamID'] +
                       [f'W{c}' for c in STAT_COLS] +
                       [f'L{c}' for c in STAT_COLS]].copy()
    w['TeamID']  = w['WTeamID']
    w['Win']     = 1
    for c in STAT_COLS:
        w[f'Off_{c}'] = w[f'W{c}']
        w[f'Def_{c}'] = w[f'L{c}']

    # Loser rows
    l            = reg[['Season','LTeamID'] +
                       [f'W{c}' for c in STAT_COLS] +
                       [f'L{c}' for c in STAT_COLS]].copy()
    l['TeamID']  = l['LTeamID']
    l['Win']     = 0
    for c in STAT_COLS:
        l[f'Off_{c}'] = l[f'L{c}']
        l[f'Def_{c}'] = l[f'W{c}']

    keep = (['Season','TeamID','Win'] +
            [f'Off_{c}' for c in STAT_COLS] +
            [f'Def_{c}' for c in STAT_COLS])

    combined = pd.concat([w[keep], l[keep]], ignore_index=True)

    agg = {'Win': ['sum', 'count']}
    for c in STAT_COLS:
        agg[f'Off_{c}'] = 'mean'
        agg[f'Def_{c}'] = 'mean'

    stats          = combined.groupby(['Season','TeamID']).agg(agg)
    stats.columns  = ['_'.join(col) for col in stats.columns]
    stats.rename(columns={'Win_sum':'NumWins','Win_count':'NumGames'}, inplace=True)
    stats['WinPct'] = stats['NumWins'] / stats['NumGames']
    stats.reset_index(inplace=True)

    # ── SEED (non-tournament teams → 17) ──────────────────────
    stats['SeedNum'] = stats.apply(
        lambda r: seed_lookup.get((r['Season'], r['TeamID']), 17), axis=1
    )
    return stats

print("\nBuilding team stats from regular season...")
team_stats  = build_team_stats(m_regular_season_detailed)
stats_index = team_stats.set_index(['Season','TeamID'])
print(f"Team-season rows : {len(team_stats):,}")

# ─────────────────────────────────────
# 4. DEFINE FEATURES
# ─────────────────────────────────────
# Per-team features: Off stats + Def stats + WinPct + Seed
TEAM_FEATURES = (
    [f'Off_{c}_mean' for c in STAT_COLS] +
    [f'Def_{c}_mean' for c in STAT_COLS] +
    ['WinPct', 'SeedNum']
)

# Model sees T1 value, T2 value, and difference for every feature
BASE_FEATURES = (
    [f'T1_{f}' for f in TEAM_FEATURES] +
    [f'T2_{f}' for f in TEAM_FEATURES] +
    [f'Diff_{f}' for f in TEAM_FEATURES]
)

# Seed-specific features added explicitly on top
SEED_FEATURES = [
    'SeedDiff',       # T1_Seed - T2_Seed  (negative = T1 is better seed)
    'AbsSeedDiff',    # absolute gap between seeds
    'UpsetFlag',      # 1 if T1 seed > T2 seed (T1 is the underdog by seed)
    'SeedProduct',    # T1_Seed x T2_Seed  (large = both are weak seeds)
    'SeedSum',        # T1_Seed + T2_Seed
]

MODEL_FEATURES = BASE_FEATURES + SEED_FEATURES

# ─────────────────────────────────────
# 5. BUILD MATCHUP ROWS
# ─────────────────────────────────────
def make_matchup_rows(games, seasons):
    rows = []
    for _, g in games[games['Season'].isin(seasons)].iterrows():
        season     = int(g['Season'])
        w, l       = int(g['WTeamID']), int(g['LTeamID'])
        t1, t2     = (w, l) if w < l else (l, w)
        label      = 1 if w == t1 else 0

        try:
            s1 = stats_index.loc[(season, t1)]
            s2 = stats_index.loc[(season, t2)]
        except KeyError:
            continue

        row = {'Season': season, 'T1_ID': t1, 'T2_ID': t2, 'Label': label}

        # Per-team + diff features
        for f in TEAM_FEATURES:
            v1 = float(s1[f]) if f in s1.index else 0.0
            v2 = float(s2[f]) if f in s2.index else 0.0
            row[f'T1_{f}']   = v1
            row[f'T2_{f}']   = v2
            row[f'Diff_{f}'] = v1 - v2

        # ── Seed-specific features ─────────────────────────────
        seed1 = float(s1['SeedNum']) if 'SeedNum' in s1.index else 17.0
        seed2 = float(s2['SeedNum']) if 'SeedNum' in s2.index else 17.0

        row['SeedDiff']    = seed1 - seed2          # negative = T1 better
        row['AbsSeedDiff'] = abs(seed1 - seed2)     # how lopsided the matchup is
        row['UpsetFlag']   = 1 if seed1 > seed2 else 0  # T1 is underdog
        row['SeedProduct'] = seed1 * seed2           # both weak → large number
        row['SeedSum']     = seed1 + seed2

        rows.append(row)
    return pd.DataFrame(rows)

# ─────────────────────────────────────
# 6. BUILD TRAIN / TEST SETS
# ─────────────────────────────────────
TRAIN_SEASONS = list(range(2013, 2023))
TEST_SEASONS  = [2023]

print("\nBuilding training matchups...")
train_reg = make_matchup_rows(m_regular_season_detailed, TRAIN_SEASONS)
train_trn = make_matchup_rows(m_tourney_detailed_result, TRAIN_SEASONS)
print(f"  Regular season : {len(train_reg):,}")
print(f"  Tournament     : {len(train_trn):,}")

train_reg['sample_weight'] = 1.0
train_trn['sample_weight'] = 3.0        # tournament games weighted higher
train_df = pd.concat([train_reg, train_trn], ignore_index=True)
print(f"  Combined total : {len(train_df):,}")

print("\nBuilding 2023 tournament test matchups...")
test_df = make_matchup_rows(m_tourney_detailed_result, TEST_SEASONS)
print(f"  Test rows      : {len(test_df)}")

# ─────────────────────────────────────
# 7. TRAIN XGBOOST
# ─────────────────────────────────────
X_train = train_df[MODEL_FEATURES].astype(float)
y_train = train_df['Label'].astype(int)
w_train = train_df['sample_weight']

X_test  = test_df[MODEL_FEATURES].astype(float)
y_test  = test_df['Label'].astype(int)

model = XGBClassifier(
    n_estimators          = 400,
    max_depth             = 4,
    learning_rate         = 0.04,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    eval_metric           = 'logloss',
    use_label_encoder     = False,
    early_stopping_rounds = 40,
    random_state          = 42,
    verbosity             = 0,
)

model.fit(
    X_train, y_train,
    sample_weight = w_train,
    eval_set      = [(X_test, y_test)],
    verbose       = False,
)

# ─────────────────────────────────────
# 8. EVALUATE
# ─────────────────────────────────────
test_df = test_df.copy()
test_df['Pred_Prob_T1_Wins'] = model.predict_proba(X_test)[:, 1]
test_df['Pred_Label']        = (test_df['Pred_Prob_T1_Wins'] >= 0.5).astype(int)
test_df['Correct']           = (test_df['Pred_Label'] == test_df['Label']).astype(int)

acc   = accuracy_score(y_test, test_df['Pred_Label'])
ll    = log_loss(y_test, test_df['Pred_Prob_T1_Wins'])
brier = brier_score_loss(y_test, test_df['Pred_Prob_T1_Wins'])

print("\n" + "="*50)
print("  2023 TOURNAMENT RESULTS")a
print("="*50)
print(f"  Accuracy    : {acc:.4f}  ({int(acc*len(test_df))}/{len(test_df)} correct)")
print(f"  Log-Loss    : {ll:.4f}")
print(f"  Brier Score : {brier:.4f}")
print("="*50)

# ─────────────────────────────────────
# 9. FEATURE IMPORTANCE
# ─────────────────────────────────────
fi = pd.DataFrame({
    'Feature'    : MODEL_FEATURES,
    'Importance' : model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)
fi['Rank'] = fi.index + 1

print("\n" + "="*68)
print("  FEATURE IMPORTANCE RANKING")
print("="*68)
print(f"  {'Rank':<5} {'Feature':<38} {'Importance':>10}  Bar")
print("-"*68)
for _, row in fi.iterrows():
    bar = '█' * int(row['Importance'] * 500)
    print(f"  {int(row['Rank']):<5} {row['Feature']:<38} {row['Importance']:>10.4f}  {bar}")
print("="*68)

# ─────────────────────────────────────
# 10. SAVE OUTPUTS
# ─────────────────────────────────────
out_dir = '/Users/shaurya/Downloads'

results = test_df[['Season','T1_ID','T2_ID',
                    'T1_SeedNum','T2_SeedNum',
                    'SeedDiff','AbsSeedDiff','UpsetFlag',
                    'Label','Pred_Prob_T1_Wins',
                    'Pred_Label','Correct']].copy()
results.rename(columns={'T1_SeedNum':'T1_Seed','T2_SeedNum':'T2_Seed'}, inplace=True)
results['Features_Used'] = ', '.join(MODEL_FEATURES)

results.to_csv(os.path.join(out_dir, 'march_mania_2023_predictions.csv'), index=False)
fi.to_csv(os.path.join(out_dir, 'march_mania_feature_importance.csv'), index=False)

print(f"\nFiles saved to {out_dir}")

# ─────────────────────────────────────
# 11. FULL FEATURE LIST SUMMARY
# ─────────────────────────────────────
print(f"\n✅ Done — {len(MODEL_FEATURES)} features total\n")
print("── BOX SCORE FEATURES (T1, T2, Diff) ──────────────────")
for f in TEAM_FEATURES:
    print(f"   T1_{f} | T2_{f} | Diff_{f}")
print("\n── SEED FEATURES ───────────────────────────────────────")
for f in SEED_FEATURES:
    print(f"   {f}")

Regular Season rows : 122,775
Seeds rows          : 2,626
Tournament rows     : 1,449

Building team stats from regular season...
Team-season rows : 8,346

Building training matchups...
  Regular season : 52,196
  Tournament     : 602
  Combined total : 52,798

Building 2023 tournament test matchups...
  Test rows      : 67

  2023 TOURNAMENT RESULTS
  Accuracy    : 0.6418  (43/67 correct)
  Log-Loss    : 0.5974
  Brier Score : 0.2042

  FEATURE IMPORTANCE RANKING
  Rank  Feature                                Importance  Bar
--------------------------------------------------------------------
  1     Diff_WinPct                                0.1422  ███████████████████████████████████████████████████████████████████████
  2     Diff_Off_Score_mean                        0.1037  ███████████████████████████████████████████████████
  3     T1_WinPct                                  0.0852  ██████████████████████████████████████████
  4     Diff_Def_Score_mean                        0.08